# RAG Pipeline — with Event Monitor

All ten pipeline stages are instrumented with `PipelineLogger`.
Events are written to `logs/rag_events.jsonl` and streamed live
to the event browser (`python event_server.py → http://localhost:5050`).

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

# ── Event logger setup ────────────────────────────────────────────────────────
from rag_logger import Stage, configure, get_logger

configure(log_dir="logs", log_file="rag_events.jsonl", level="INFO", console=True)
_log = get_logger("notebook")
_log.info("Notebook started — RAG pipeline initialising")
print("Logger configured → logs/rag_events.jsonl")

In [ ]:
DOCUMENTS = [
    """Python is a high-level, interpreted programming language created by Guido van Rossum
    and first released in 1991. Python's design philosophy emphasizes code readability,
    and its syntax allows programmers to express concepts in fewer lines of code than
    languages like C++ or Java. Python supports multiple programming paradigms, including
    structured, object-oriented, and functional programming.""",

    """Python is widely used in data science, machine learning, and artificial intelligence.
    Libraries like NumPy, Pandas, and Matplotlib form the foundation of the scientific
    Python ecosystem. For machine learning, popular frameworks include scikit-learn,
    TensorFlow, PyTorch, and Keras. Python is also the dominant language for deep
    learning research and production deployments.""",

    """Python's package manager, pip, allows users to install thousands of third-party
    packages from the Python Package Index (PyPI). Virtual environments, managed with
    tools like venv or conda, let developers isolate project dependencies. Python 3.x
    is the current active version; Python 2 reached end-of-life in January 2020.""",

    """FastAPI is a modern, fast web framework for building APIs with Python 3.7+ based on
    standard Python type hints. It is one of the fastest Python web frameworks available,
    on par with NodeJS and Go. FastAPI automatically generates interactive API documentation
    using Swagger UI and ReDoc. It uses Pydantic for data validation.""",

    """Django is a high-level Python web framework that encourages rapid development and
    clean, pragmatic design. Built by experienced developers, it takes care of much of
    the hassle of web development so you can focus on writing your app. Django follows
    the model-view-template (MVT) architectural pattern and includes an ORM, admin
    interface, authentication system, and many other built-in features.""",

    """The Global Interpreter Lock (GIL) in CPython is a mutex that protects access to
    Python objects, preventing multiple threads from executing Python bytecodes at once.
    This simplifies memory management but limits multi-threaded CPU-bound performance.
    For CPU-bound parallelism, Python developers typically use the multiprocessing module
    or external libraries like Dask or Ray instead of threading.""",

    """Python decorators are a powerful and expressive feature that allows modification of
    functions or classes. A decorator is a function that takes another function as input
    and returns a modified version of it. Common built-in decorators include @staticmethod,
    @classmethod, and @property. Decorators are heavily used in frameworks like Flask and
    FastAPI for routing and middleware.""",

    """Asynchronous programming in Python is supported through the asyncio library,
    introduced in Python 3.4. The async/await syntax allows writing concurrent code
    that is easy to read. This is especially useful for I/O-bound tasks like web requests,
    database queries, and file operations. Frameworks like aiohttp and FastAPI are built
    on top of asyncio for high-performance async web services.""",
]

_log = get_logger(Stage.DATA_INGESTION)
_log.info(
    "Documents corpus loaded",
    extra={"doc_count": len(DOCUMENTS), "source": "in-memory corpus"}
)
print(f"Loaded {len(DOCUMENTS)} documents")

In [ ]:
EVAL_DATASET = [
    {
        "question": "Who created Python and when was it first released?",
        "ground_truth": "Python was created by Guido van Rossum and first released in 1991.",
    },
    {
        "question": "What is the GIL in Python and how does it affect multithreading?",
        "ground_truth": (
            "The GIL (Global Interpreter Lock) in CPython is a mutex that prevents multiple "
            "threads from executing Python bytecodes simultaneously, limiting CPU-bound "
            "multi-threaded performance. Developers use multiprocessing or libraries like "
            "Dask for CPU-bound parallelism."
        ),
    },
    {
        "question": "What are Python decorators and give some examples?",
        "ground_truth": (
            "Python decorators modify functions or classes by wrapping them. Built-in examples "
            "include @staticmethod, @classmethod, and @property. They are used heavily in "
            "frameworks like Flask and FastAPI."
        ),
    },
    {
        "question": "What is FastAPI and what are its key features?",
        "ground_truth": (
            "FastAPI is a modern Python web framework for building APIs using type hints. "
            "It is very fast, auto-generates Swagger/ReDoc documentation, and uses Pydantic "
            "for data validation."
        ),
    },
    {
        "question": "How does async programming work in Python?",
        "ground_truth": (
            "Python supports async programming via the asyncio library (Python 3.4+) using "
            "async/await syntax. It is best for I/O-bound tasks. Frameworks like aiohttp and "
            "FastAPI are built on asyncio."
        ),
    },
]

_log = get_logger(Stage.DATA_INGESTION)
_log.info(
    "Evaluation dataset ready",
    extra={"eval_questions": len(EVAL_DATASET)}
)
print(f"Eval dataset: {len(EVAL_DATASET)} questions")

In [ ]:
import time
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# ── STAGE: Chunking ──────────────────────────────────────────────────────────
_log_chunk = get_logger(Stage.CHUNKING)
_log_chunk.info(
    "Starting chunking",
    extra={"docs": len(DOCUMENTS), "chunk_size": 400, "overlap": 50}
)

docs = [Document(page_content=text) for text in DOCUMENTS]

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
splits = splitter.split_documents(docs)

# warn if any chunk exceeds a safe token estimate
oversized = [s for s in splits if len(s.page_content.split()) > 350]
if oversized:
    _log_chunk.warning(
        f"Oversized chunks detected — {len(oversized)} chunks may exceed token limit",
        extra={"oversized_count": len(oversized), "threshold_words": 350}
    )

_log_chunk.info(
    "Chunking complete",
    extra={"total_chunks": len(splits), "oversized": len(oversized)}
)
print(f"Total chunks created: {len(splits)}")

# ── STAGE: Embedding ─────────────────────────────────────────────────────────
_log_emb = get_logger(Stage.EMBEDDING)
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
_log_emb.info(
    "Loading embedding model",
    extra={"model": EMBED_MODEL, "device": "cpu"}
)

t0 = time.perf_counter()
try:
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cpu"},
    )
    load_time = round(time.perf_counter() - t0, 2)
    _log_emb.info(
        "Embedding model loaded",
        extra={"model": EMBED_MODEL, "load_time_s": load_time}
    )
except Exception as exc:
    _log_emb.critical(
        "Embedding model failed to load",
        extra={"model": EMBED_MODEL, "error": str(exc)}
    )
    raise

# ── STAGE: Vector Indexing ───────────────────────────────────────────────────
_log_idx = get_logger(Stage.VECTOR_INDEXING)
_log_idx.info(
    "Building FAISS vector index",
    extra={"vectors": len(splits), "index": "FAISS-in-memory"}
)

t0 = time.perf_counter()
try:
    vectorstore = FAISS.from_documents(splits, embeddings)
    index_time = round(time.perf_counter() - t0, 2)
    _log_idx.info(
        "Vector index built successfully",
        extra={"vectors_indexed": len(splits), "build_time_s": index_time}
    )
except Exception as exc:
    _log_idx.error(
        "FAISS index build failed",
        extra={"error": str(exc)}
    )
    raise

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

_log_idx.info(
    "Retriever ready",
    extra={"top_k": 3, "index": "FAISS-in-memory"}
)
print("Vector store ready")

In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# ── STAGE: LLM Generation — model init ──────────────────────────────────────
_log_llm = get_logger(Stage.LLM_GENERATION)
LLM_MODEL = "llama-3.3-70b-versatile"
_log_llm.info(
    "Initialising LLM",
    extra={"model": LLM_MODEL, "temperature": 0.4, "provider": "Groq"}
)

try:
    llm = ChatGroq(
        api_key=GROQ_API_KEY,
        model_name=LLM_MODEL,
        temperature=0.4,
    )
    _log_llm.info("LLM initialised", extra={"model": LLM_MODEL})
except Exception as exc:
    _log_llm.critical(
        "LLM initialisation failed",
        extra={"model": LLM_MODEL, "error": str(exc)}
    )
    raise

# ── STAGE: Context Assembly — prompt template ────────────────────────────────
_log_ctx = get_logger(Stage.CONTEXT_ASSEMBLY)
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""
Use the following context to answer the question concisely and accurately.

Context:
{context}

Question:
{question}

Answer:
""",
)
_log_ctx.info(
    "Prompt template created",
    extra={"input_variables": ["context", "question"], "chain_type": "stuff"}
)

# ── RAG Chain ────────────────────────────────────────────────────────────────
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt_template},
)

_log_llm.info("RAG chain assembled", extra={"chain_type": "RetrievalQA/stuff"})
print("RAG Chain Ready")

In [ ]:
import uuid

results = []

_log_ret  = get_logger(Stage.RETRIEVAL)
_log_ctx  = get_logger(Stage.CONTEXT_ASSEMBLY)
_log_llm  = get_logger(Stage.LLM_GENERATION)
_log_val  = get_logger(Stage.OUTPUT_VALIDATION)

for item in EVAL_DATASET:
    question   = item["question"]
    request_id = f"REQ-{uuid.uuid4().hex[:6].upper()}"

    # ── Retrieval ────────────────────────────────────────────────────────────
    _log_ret.info(
        "Retrieving context for query",
        extra={"request_id": request_id, "query_preview": question[:80]}
    )

    # ── LLM Generation ───────────────────────────────────────────────────────
    _log_llm.info(
        "Sending query to LLM",
        extra={"request_id": request_id, "model": LLM_MODEL}
    )

    t0 = time.perf_counter()
    try:
        response = rag_chain.invoke({"query": question})
        elapsed  = round(time.perf_counter() - t0, 2)
    except Exception as exc:
        _log_llm.critical(
            "LLM inference failed",
            extra={"request_id": request_id, "error": str(exc)}
        )
        raise

    answer   = response["result"]
    src_docs = response["source_documents"]
    contexts = [doc.page_content for doc in src_docs]

    if elapsed > 10:
        _log_llm.warning(
            f"LLM response slow — {elapsed}s",
            extra={"request_id": request_id, "elapsed_s": elapsed}
        )
    else:
        _log_llm.info(
            "LLM response received",
            extra={"request_id": request_id, "elapsed_s": elapsed}
        )

    # ── Context Assembly — log retrieved sources ──────────────────────────────
    total_ctx_words = sum(len(c.split()) for c in contexts)
    if total_ctx_words > 600:
        _log_ctx.warning(
            "Assembled context is large",
            extra={"request_id": request_id, "context_words": total_ctx_words}
        )
    else:
        _log_ctx.info(
            "Context assembled",
            extra={
                "request_id": request_id,
                "chunks_used": len(contexts),
                "context_words": total_ctx_words,
            }
        )

    # ── Output Validation ────────────────────────────────────────────────────
    if not answer or not answer.strip():
        _log_val.error(
            "Empty answer returned by LLM",
            extra={"request_id": request_id, "question_preview": question[:60]}
        )
    elif len(answer.split()) < 5:
        _log_val.warning(
            "Answer suspiciously short",
            extra={"request_id": request_id, "word_count": len(answer.split())}
        )
    else:
        _log_val.info(
            "Output validation passed",
            extra={"request_id": request_id, "answer_words": len(answer.split())}
        )

    results.append({
        "question":     question,
        "answer":       answer,
        "contexts":     contexts,
        "ground_truth": item["ground_truth"],
    })

    print(f"\nQ: {question}")
    print(f"A: {answer}")

In [ ]:
from datasets import Dataset

_log_dep = get_logger(Stage.DEPLOYMENT)

ragas_data = {
    "question":    [r["question"]    for r in results],
    "answer":      [r["answer"]      for r in results],
    "contexts":    [r["contexts"]    for r in results],
    "ground_truth":[r["ground_truth"]for r in results],
}

dataset = Dataset.from_dict(ragas_data)

_log_dep.info(
    "RAGAS evaluation dataset assembled",
    extra={"rows": len(dataset), "columns": list(ragas_data.keys())}
)

dataset

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

_log_mon = get_logger(Stage.MONITORING)
_log_mon.info(
    "Starting RAGAS evaluation",
    extra={"metrics": ["faithfulness","answer_relevancy","context_precision","context_recall"]}
)

ragas_llm        = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(embeddings)

metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

for metric in metrics:
    metric.llm = ragas_llm
    if hasattr(metric, "embeddings"):
        metric.embeddings = ragas_embeddings

t0 = time.perf_counter()
try:
    scores = evaluate(dataset=dataset, metrics=metrics)
    eval_time = round(time.perf_counter() - t0, 2)
    _log_mon.info(
        "RAGAS evaluation complete",
        extra={"eval_time_s": eval_time, "questions_evaluated": len(dataset)}
    )
except Exception as exc:
    _log_mon.error(
        "RAGAS evaluation failed",
        extra={"error": str(exc)}
    )
    raise

print("Evaluation Complete")

In [ ]:
import pandas as pd

scores_df = scores.to_pandas()

metric_cols = ["faithfulness","answer_relevancy","context_precision","context_recall"]
summary     = scores_df[metric_cols].mean()

_log_mon = get_logger(Stage.MONITORING)

# log per-metric scores and warn on anything below 0.7
for metric, val in summary.items():
    rounded = round(float(val), 3)
    if rounded < 0.7:
        _log_mon.warning(
            f"Low RAGAS score — {metric}: {rounded}",
            extra={"metric": metric, "score": rounded, "threshold": 0.7}
        )
    else:
        _log_mon.info(
            f"RAGAS metric OK — {metric}: {rounded}",
            extra={"metric": metric, "score": rounded}
        )

overall = round(float(summary.mean()), 3)
status  = "healthy" if overall >= 0.7 else "degraded"
_log_mon.info(
    f"Pipeline health report — overall RAGAS score: {overall} ({status})",
    extra={"overall_score": overall, "status": status}
)

scores_df

In [ ]:
print("\n=== RAGAS Score Summary ===")
print(summary.to_string())

output_path = "ragas_results.csv"
scores_df.to_csv(output_path, index=False)

_log_mon = get_logger(Stage.MONITORING)
_log_mon.info(
    "Results saved to disk",
    extra={"path": output_path, "rows": len(scores_df)}
)

from rag_logger import get_log_path
print(f"\nResults saved to: {output_path}")
print(f"Event log written to: {get_log_path()}")
print("Open the event browser: python event_server.py  →  http://localhost:5050")